# ONNX Runtime CPU EP Performance Analysis — FP16 QDQ Models

**Objective:** Compare ORT CPU EP latency across quantization formats (MNB native, QDQ fused, QDQ unfused), bit widths (4, 8), symmetry/signedness/granularity variants across devices (AMD, Intel, ARM) for **FP16 accumulation** models.

**Devices:**
| Label | CPU | ORT Version | Notes |
|-------|-----|-------------|-------|
| `amd` | AMD64 Family 26 Model 36 (HP AMD laptop) | 1.25.0 | 10 iterations |
| `intel` | Intel64 Family 6 Model 186 (Surface Laptop) | 1.24.1 | 10 iterations |
| `arm` | ARM64 Qualcomm (Surface ARM laptop) | 1.25.0.dev20260224003 | 10 iterations |

**Notes:**
- All models use `original` weight layout only — no transpose variants in this dataset
- ORT only fuses DQ+MatMul → MatMulNBits for **4-bit block** quantization
- `channel-sym-signed` QDQ models fail to load on all devices (same ORT bug as FP32)
- AMD and ARM have limited 7b QDQ coverage (only 4-bit block asym signed fused/unfused); Intel has full 7b QDQ coverage across all variants
- ARM 8-bit MNB shows anomalously high latency (~900-1000ms at seq=1 for 0.5b) — flagged in observations
- FP32 comparison data loaded from `qdq-cpu/` for cross-precision analysis (Section K)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

# --- Load all three devices (FP16) ---
DATA_DIR = Path(r"C:\dev\qdq-cpu-fp16")
DEVICES = {
    "amd": DATA_DIR / "hp_amd_laptop_perf",
    "intel": DATA_DIR / "intel_surface_laptop_perf",
    "arm": DATA_DIR / "surface_arm_laptop_perf",
}

frames = []
for device, path in DEVICES.items():
    csv_path = path / "summary.csv"
    if not csv_path.exists():
        print(f"WARNING: {csv_path} not found, skipping")
        continue
    tmp = pd.read_csv(csv_path)
    tmp["device"] = device
    frames.append(tmp)
    print(f"  {device}: {len(tmp)} rows from {csv_path}")

df = pd.concat(frames, ignore_index=True)

# Type coercion
df["bits"] = df["bits"].astype(int)
df["seq_length"] = df["seq_length"].astype(int)
df["mean_ms"] = df["mean_ms"].astype(float)
df["model_load_time_s"] = pd.to_numeric(df["model_load_time_s"], errors="coerce")

# Fill MNB granularity/signedness
mnb_mask = df["format_type"] == "mnb"
df.loc[mnb_mask, "granularity"] = "n/a"
df.loc[mnb_mask, "signedness"] = "n/a"

# Save combined CSV
out_path = DATA_DIR / "all_devices_summary.csv"
df.to_csv(out_path, index=False)
print(f"\nSaved combined CSV to {out_path}")

print(f"\nTotal rows: {len(df)}")
print(f"Devices: {list(df['device'].unique())}")
print(f"Weight layouts: {list(df['weight_layout'].unique())}")
print(f"Scenarios: {list(df['scenario'].unique())}")
print(f"Seq lengths: {sorted(int(x) for x in df['seq_length'].unique())}")
print(f"Bits: {sorted(int(x) for x in df['bits'].unique())}")
print(f"Model sizes: {list(df['model_size'].unique())}")
print(f"Format types: {list(df['format_type'].unique())}")
print(f"Granularities: {list(df['granularity'].unique())}")

  amd: 285 rows from ..\..\..\data\qdq-cpu-fp16\hp_amd_laptop_perf\summary.csv
  intel: 340 rows from ..\..\..\data\qdq-cpu-fp16\intel_surface_laptop_perf\summary.csv
  arm: 285 rows from ..\..\..\data\qdq-cpu-fp16\surface_arm_laptop_perf\summary.csv

Saved combined CSV to ..\..\..\data\qdq-cpu-fp16\all_devices_summary.csv

Total rows: 910
Devices: ['amd', 'intel', 'arm']
Weight layouts: ['original']
Scenarios: ['native', 'qdq_fused', 'qdq_unfused']
Seq lengths: [1, 128, 256, 512, 1024]
Bits: [4, 8]
Model sizes: ['0.5b', '1.5b', '3b', '7b']
Format types: ['mnb', 'qdq']
Granularities: ['n/a', 'block', 'channel']


In [2]:
# --- Helper functions ---
MODEL_SIZE_ORDER = ["0.5b", "1.5b", "3b", "7b"]
SEQ_ORDER = [1, 128, 256, 512, 1024]
DEVICE_ORDER = ["amd", "intel", "arm"]

def pivot_latency(data, index_cols, columns_col, values_col="mean_ms"):
    """Pivot data into a wide table with mean_ms values."""
    return data.pivot_table(
        index=index_cols, columns=columns_col,
        values=values_col, aggfunc="first"
    )


def ratio_table(data, col_a, col_b, label_a="A", label_b="B"):
    """Compute ratio B/A for two subsets identified by a column value."""
    a = data[data[col_a] == label_a].copy()
    b = data[data[col_a] == label_b].copy()
    merge_cols = [c for c in a.columns if c not in ["mean_ms", col_a, "model_load_time_s"]]
    merged = a.merge(b, on=merge_cols, suffixes=(f"_{label_a}", f"_{label_b}"))
    merged["ratio"] = merged[f"mean_ms_{label_b}"] / merged[f"mean_ms_{label_a}"]
    return merged


def comparison_table(data, group_cols, compare_col, compare_values, metric="mean_ms"):
    """Build a side-by-side comparison table.

    Returns a DataFrame with one row per (group_cols) combination and
    one column per compare_value, plus a ratio column if exactly 2 compare_values.
    """
    tables = {}
    for val in compare_values:
        subset = data[data[compare_col] == val].set_index(group_cols)[metric]
        tables[val] = subset

    result = pd.DataFrame(tables)
    if len(compare_values) == 2:
        a, b = compare_values
        result[f"{b}/{a}"] = result[b] / result[a]
    return result


def display_comparison(title, data, group_cols, compare_col, compare_values, metric="mean_ms"):
    """Print a comparison table with a title."""
    print(f"\n{'='*80}")
    print(f"  {title}")
    print(f"{'='*80}")
    tbl = comparison_table(data, group_cols, compare_col, compare_values, metric)
    print(tbl.to_string(float_format=lambda x: f"{x:.2f}"))
    return tbl

## A. MNB Native vs QDQ Fused vs QDQ Unfused — 4-bit block quantization

**Expected**: MNB ≈ QDQ fused << QDQ unfused (same as FP32)

In [3]:
# Filter to 4-bit, block-quantized, original layout, asym signed (most common LLM config)
mnb_4 = df[(df["format_type"] == "mnb") & (df["bits"] == 4) & (df["weight_layout"] == "original")].copy()
qdq_fused_4 = df[
    (df["format_type"] == "qdq") & (df["bits"] == 4) & (df["granularity"] == "block") &
    (df["weight_layout"] == "original") & (df["scenario"] == "qdq_fused") & (df["signedness"] == "signed")
].copy()
qdq_unfused_4 = df[
    (df["format_type"] == "qdq") & (df["bits"] == 4) & (df["granularity"] == "block") &
    (df["weight_layout"] == "original") & (df["scenario"] == "qdq_unfused") & (df["signedness"] == "signed")
].copy()

# Build merged comparison table (asym only — sym behaves identically)
mnb_sub = mnb_4[mnb_4["symmetry"] == "asym"][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "mnb_ms"})
fused_sub = qdq_fused_4[qdq_fused_4["symmetry"] == "asym"][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "fused_ms"})
unfused_sub = qdq_unfused_4[qdq_unfused_4["symmetry"] == "asym"][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "unfused_ms"})

merged = mnb_sub.merge(fused_sub, on=["device", "model_size", "seq_length"], how="outer")
merged = merged.merge(unfused_sub, on=["device", "model_size", "seq_length"], how="outer")
merged["fused/mnb"] = merged["fused_ms"] / merged["mnb_ms"]
merged["unfused/mnb"] = merged["unfused_ms"] / merged["mnb_ms"]
merged = merged.set_index(["device", "model_size", "seq_length"]).sort_index()

# --- Table 1: Latency (ms) — seq=1 (decode) and seq=1024 (prefill) ---
print("  Table 1: Latency comparison (ms) — 4-bit block asym signed, original layout")
print("  " + "-"*85)
for seq in [1, 1024]:
    label = "Decode (seq=1)" if seq == 1 else "Prefill (seq=1024)"
    sub = merged.xs(seq, level="seq_length")[["mnb_ms", "fused_ms", "unfused_ms", "fused/mnb", "unfused/mnb"]]
    print(f"\n  {label}:")
    print(sub.to_string(float_format=lambda x: f"{x:.1f}"))

# --- Table 2: Model load times ---
mnb_load = mnb_4[mnb_4["symmetry"] == "asym"].drop_duplicates(subset=["device", "model_size"])[["device", "model_size", "model_load_time_s"]].rename(columns={"model_load_time_s": "mnb_s"})
fused_load = qdq_fused_4[qdq_fused_4["symmetry"] == "asym"].drop_duplicates(subset=["device", "model_size"])[["device", "model_size", "model_load_time_s"]].rename(columns={"model_load_time_s": "fused_s"})
unfused_load = qdq_unfused_4[qdq_unfused_4["symmetry"] == "asym"].drop_duplicates(subset=["device", "model_size"])[["device", "model_size", "model_load_time_s"]].rename(columns={"model_load_time_s": "unfused_s"})
load_times = mnb_load.merge(fused_load, on=["device", "model_size"], how="outer").merge(unfused_load, on=["device", "model_size"], how="outer")
load_times["fused/mnb"] = load_times["fused_s"] / load_times["mnb_s"]

print(f"\n\n  Table 2: Model load times (s) — includes PrePack for MNB & fused; unfused skips PrePack")
print("  " + "-"*85)
print(load_times.set_index(["device", "model_size"]).sort_index().to_string(float_format=lambda x: f"{x:.2f}"))

  Table 1: Latency comparison (ms) — 4-bit block asym signed, original layout
  -------------------------------------------------------------------------------------

  Decode (seq=1):
                   mnb_ms  fused_ms  unfused_ms  fused/mnb  unfused/mnb
device model_size                                                      
amd    0.5b           9.3    2871.9      3071.6      308.8        330.3
       1.5b          23.2    7938.8      8204.5      341.5        353.0
       3b            45.2   19779.3     20931.0      437.1        462.6
       7b            88.1   45564.7     47337.6      517.1        537.2
arm    0.5b           9.7    2130.7      2264.8      219.6        233.4
       1.5b          39.8    7642.3      7642.2      191.8        191.8
       3b            75.6   67694.5     45704.7      895.2        604.4
       7b           118.4  355211.6    376837.0     2999.0       3181.6
intel  0.5b          10.7    3363.6      3282.2      315.5        307.9
       1.5b          26

In [4]:
# --- Table 3: Estimated DequantizeLinear overhead (unfused - fused) at seq=1 ---
m = merged.copy()
m["dq_overhead_ms"] = m["unfused_ms"] - m["fused_ms"]

seq1 = m.xs(1, level="seq_length")[["fused_ms", "unfused_ms", "dq_overhead_ms"]]

print("  Table 3: Estimated DequantizeLinear cost at seq=1 (ms)")
print("  At decode (seq=1), MatMul compute is trivial → delta ≈ pure DQ overhead")
print("  " + "-"*70)
print(seq1.to_string(float_format=lambda x: f"{x:.0f}"))

print("\n\n  DQ cost scaling relative to 0.5b (should be ~linear with param count):")
for dev in DEVICE_ORDER:
    try:
        dd = seq1.xs(dev, level="device")["dq_overhead_ms"]
        base = dd.iloc[0]
        if base > 0:
            print(f"    {dev}: " + "  ".join(f"{sz}={v/base:.1f}x" for sz, v in dd.items()))
    except KeyError:
        pass

  Table 3: Estimated DequantizeLinear cost at seq=1 (ms)
  At decode (seq=1), MatMul compute is trivial → delta ≈ pure DQ overhead
  ----------------------------------------------------------------------
                   fused_ms  unfused_ms  dq_overhead_ms
device model_size                                      
amd    0.5b            2872        3072             200
       1.5b            7939        8204             266
       3b             19779       20931            1152
       7b             45565       47338            1773
arm    0.5b            2131        2265             134
       1.5b            7642        7642              -0
       3b             67695       45705          -21990
       7b            355212      376837           21625
intel  0.5b            3364        3282             -81
       1.5b           10495       10114            -381
       3b             21129       21149              20
       7b             60658       58750           -1909


  DQ cost 

### Observations

1. **FP16 QDQ fusion is completely broken.** At seq=1, the "fused" QDQ path is 192–3000× slower than native MNB. AMD 309–517×, Intel 316–566×, ARM 192–2999×. In the FP32 notebook, fused QDQ matched MNB at ~1.0×.

2. **Fused ≈ unfused.** The fused and unfused QDQ paths produce nearly identical latencies (within noise). Both are running the same naïve DQ+MatMul — the graph rewrite never fires.

3. **Table 3 DQ overhead is meaningless here.** Because neither path fuses, the fused vs unfused delta is just noise. **WARNING:** ARM 3b shows dq_overhead = −21,990ms and ARM 7b = +21,625ms — these are not noise but garbage data from the ARM 3b/7b instability issue (see Section C). The DQ scaling ratios (arm: 3b=−164.0x, 7b=161.3x) are nonsensical and should be ignored. In FP32, this delta cleanly isolates the DQ kernel cost.

4. **Load time confirms no fusion.** QDQ "fused" models load in 0.25–0.47s vs MNB 0.59–11.3s. No PrePack overhead means no weight repacking means no fusion. In FP32, fused QDQ load times matched MNB (both do PrePack).

5. **Root cause:** The `DQMatMulToMatMulNBits` graph rewrite in ORT requires DequantizeLinear to output fp32 tensors. FP16 accumulation models produce fp16 DQ outputs, so the pattern never matches. The selector's type check rejects every FP16 QDQ model.

6. At seq=1024, the gap shrinks but remains large: AMD 2.5–6.9×, Intel 1.5–3.1×, ARM 1.3–11.8×. Even at prefill the unfused DQ kernel dominates.

---

## B. MNB Symmetric vs Asymmetric

Check if asymmetric (with zero-point) adds overhead vs symmetric.

In [5]:
mnb = df[(df["format_type"] == "mnb") & (df["weight_layout"] == "original")].copy()

for bits in [4, 8]:
    sub = mnb[mnb["bits"] == bits]
    tbl = comparison_table(
        sub,
        group_cols=["device", "model_size", "seq_length"],
        compare_col="symmetry",
        compare_values=["sym", "asym"],
    )
    tbl.columns = ["sym", "asym", "asym/sym"]

    print(f"\n{'='*90}")
    print(f"  B. MNB sym vs asym — {bits}-bit (ratio = asym/sym)")
    print(f"{'='*90}")
    print(tbl.to_string(float_format=lambda x: f"{x:.2f}"))

    # Summary stats per device
    print(f"\n  Summary:")
    for dev in DEVICE_ORDER:
        try:
            ratios = tbl.loc[dev]["asym/sym"]
            print(f"    {dev}: min={ratios.min():.2f}  max={ratios.max():.2f}  "
                  f"median={ratios.median():.2f}  mean={ratios.mean():.2f}  std={ratios.std():.2f}")
        except KeyError:
            pass

# ARM 4-bit: check if the difference is systematic per model size
print(f"\n{'='*90}")
print("  ARM 4-bit asym/sym by model_size")
print(f"{'='*90}")
arm_4 = mnb[(mnb["device"] == "arm") & (mnb["bits"] == 4)]
tbl_arm = comparison_table(arm_4, ["model_size", "seq_length"], "symmetry", ["sym", "asym"])
tbl_arm.columns = ["sym", "asym", "asym/sym"]
for ms in MODEL_SIZE_ORDER:
    if ms not in tbl_arm.index.get_level_values(0):
        continue
    ratios = tbl_arm.loc[ms]["asym/sym"]
    mean_r = ratios.mean()
    direction = "asym SLOWER" if mean_r > 1.05 else ("asym FASTER" if mean_r < 0.95 else "~EQUAL")
    print(f"  {ms}: mean={mean_r:.2f} ({direction})  per-seq={[f'{v:.2f}' for v in ratios.values]}")


  B. MNB sym vs asym — 4-bit (ratio = asym/sym)
                                  sym     asym  asym/sym
device model_size seq_length                            
amd    0.5b       1              8.85     9.30      1.05
                  128          197.82   169.48      0.86
                  256          405.41   347.58      0.86
                  512          959.67   970.28      1.01
                  1024        1953.81  1976.00      1.01
       1.5b       1             22.72    23.25      1.02
                  128          521.75   519.36      1.00
                  256         1194.05  1219.35      1.02
                  512         2707.22  2312.94      0.85
                  1024        5097.96  4925.51      0.97
       3b         1             42.15    45.25      1.07
                  128         1119.99  1136.42      1.01
                  256         2442.83  2572.79      1.05
                  512         4685.69  4850.28      1.04
                  1024       10253.56 1

### Observations

1. **x86: sym ≈ asym across both bit widths.** AMD 4-bit median 0.99 (range 0.93–1.17), Intel 4-bit median 1.00 (0.89–1.14). 8-bit: AMD median 1.00, Intel median 0.99. Same finding as FP32 — zero-point handling has no measurable cost on x86.

2. **ARM 4-bit:** median 1.00 (range 0.74–1.39). Wider spread than x86 but no systematic directional bias. Unlike FP32 where ARM showed a consistent ~50% asym penalty for models ≥1.5b, FP16 MNB shows no such penalty. This suggests the ARM asym issue from FP32 may be specific to the fp32 MNB kernel path.

3. **ARM 8-bit:** median 0.97 (range 0.85–1.06). Near parity, cleaner than the erratic ARM 8-bit ratios seen in FP32. Likely because the ARM 8-bit FP16 MNB kernel takes a different (much slower) code path that happens to handle sym/asym uniformly.

Conclusion: Symmetry type is a non-factor for FP16 MNB on all devices. The ARM asym penalty from FP32 does not reproduce in FP16.

---

## C. MNB 4-bit vs 8-bit

Quantify latency difference between 4-bit and 8-bit. 4-bit should be faster at decode (less bandwidth). At prefill, compute dominates so difference should be small.

**Note:** ARM 8-bit MNB shows anomalously high seq=1 latency — flag in observations.

In [6]:
# Section C: MNB 4-bit vs 8-bit
# Section B showed sym ≈ asym on x86. Use sym here (cleaner data).
mnb = df[(df["format_type"] == "mnb") & (df["weight_layout"] == "original")].copy()
mnb_sym = mnb[mnb["symmetry"] == "sym"]

tbl = comparison_table(
    mnb_sym,
    group_cols=["device", "model_size", "seq_length"],
    compare_col="bits",
    compare_values=[4, 8],
)
tbl.columns = ["4-bit", "8-bit", "8b/4b"]

print(f"{'='*80}")
print(f"  C. MNB 4-bit vs 8-bit — sym, original layout (ratio = 8b/4b)")
print(f"{'='*80}")
print(tbl.to_string(float_format=lambda x: f"{x:.2f}"))

# Verify asym ratios match on x86
tbl_asym = comparison_table(
    mnb[(mnb["symmetry"] == "asym") & (mnb["device"].isin(["amd", "intel"]))],
    group_cols=["device", "model_size", "seq_length"],
    compare_col="bits",
    compare_values=[4, 8],
)
tbl_asym.columns = ["4-bit", "8-bit", "8b/4b"]
ratio_diff = (tbl_asym["8b/4b"] - tbl.loc[["amd", "intel"]]["8b/4b"]).abs()
print(f"\nAsym vs sym ratio check (x86): max diff = {ratio_diff.max():.2f}, mean = {ratio_diff.mean():.2f}")

  C. MNB 4-bit vs 8-bit — sym, original layout (ratio = 8b/4b)
                                4-bit    8-bit  8b/4b
device model_size seq_length                         
amd    0.5b       1              8.85    11.89   1.34
                  128          197.82   184.01   0.93
                  256          405.41   350.70   0.87
                  512          959.67   867.65   0.90
                  1024        1953.81  1749.95   0.90
       1.5b       1             22.72    34.64   1.52
                  128          521.75   485.89   0.93
                  256         1194.05  1113.77   0.93
                  512         2707.22  2207.88   0.82
                  1024        5097.96  4826.04   0.95
       3b         1             42.15    63.32   1.50
                  128         1119.99   917.32   0.82
                  256         2442.83  2075.74   0.85
                  512         4685.69  4075.33   0.87
                  1024       10253.56  9540.76   0.93
       7b         1

### Observations

1. **x86 seq=1:** 8b/4b = 1.3–1.8×. Expected — bandwidth-bound, 4-bit weights are half the size. Same direction and magnitude as FP32 (1.4–2.0×).

2. **x86 seq≥128:** gap narrows to ~0.9–1.0× as compute dominates. AMD 8-bit becomes slightly faster at seq=1024 (0.9×), Intel stays near 1.0×. Consistent with FP32.

3. **ARM 8-bit is catastrophically slow at seq=1.** 8b/4b = 67–124× (0.5b=67×, 1.5b=124×, 3b=95×, 7b=104×). This is NOT a memory issue — the 0.5b model is tiny. Something specific to the ARM FP16 8-bit MNB kernel is broken.

4. **ARM 8-bit load time is suspiciously fast:** 0.16–0.27s vs 4-bit's 0.70–8.13s. This suggests the ARM 8-bit FP16 MNB kernel is NOT doing PrePack (weight repacking). The combination of near-instant load + 67–124× slow execution strongly implies the kernel falls back to a naïve unoptimized path.

5. **ARM seq=1024:** gap shrinks to 1.3–1.5×. The catastrophic overhead at short sequences amortizes at prefill, but 8-bit is still slower. In FP32, ARM 8-bit was comparable to 4-bit at prefill.

**⚠️ ARM 3b/7b Data Quality Warning:** Throughout sections C–K, ARM 3b and 7b models show erratic behavior in QDQ scenarios (e.g., negative DQ overheads, 2× inversions). These anomalies likely stem from memory pressure on the ARM device, not kernel bugs. ARM 3b/7b QDQ results should be treated as unreliable.

Conclusion: FP16 4-bit MNB performs as expected across all devices. FP16 8-bit MNB has a severe ARM-specific bug — likely missing PrePack/SIMD for the fp16 8-bit kernel path.

---

## D. QDQ Signed vs Unsigned Asymmetric

Check if signed vs unsigned affects QDQ performance. Since FP16 fusion is broken (Section A), both 4-bit and 8-bit run unfused DQ+MatMul. Expected: ~1.0x.

In [7]:
# Section D: QDQ signed vs unsigned — asymmetric, block, original layout
# 4-bit: check both fused (MatMulNBits) and unfused (DequantizeLinear) paths
# 8-bit: only fused scenario (which is actually unfused — 8-bit doesn't fuse)

qdq_asym_block_orig = df[
    (df["format_type"] == "qdq") & (df["symmetry"] == "asym") & (df["granularity"] == "block") &
    (df["weight_layout"] == "original")
].copy()

configs = [
    (4, "qdq_fused",   "4-bit fused (MatMulNBits kernel)"),
    (4, "qdq_unfused", "4-bit unfused (DequantizeLinear + MatMul)"),
    (8, "qdq_fused",   "8-bit (no fusion — runs DequantizeLinear + MatMul)"),
]

for bits, scenario, label in configs:
    sub = qdq_asym_block_orig[(qdq_asym_block_orig["bits"] == bits) & (qdq_asym_block_orig["scenario"] == scenario)]
    if sub.empty:
        print(f"\n  {label}: No data available")
        continue
    tbl = comparison_table(
        sub,
        group_cols=["device", "model_size", "seq_length"],
        compare_col="signedness",
        compare_values=["signed", "unsigned"],
    )
    tbl.columns = ["signed", "unsigned", "u/s"]

    print(f"\n{'='*90}")
    print(f"  D. {label} — unsigned/signed ratio")
    print(f"{'='*90}")
    for dev in DEVICE_ORDER:
        try:
            ratios = tbl.loc[dev]["u/s"]
            print(f"  {dev}: min={ratios.min():.2f}  max={ratios.max():.2f}  median={ratios.median():.2f}")
        except KeyError:
            pass
    # Show outliers
    outliers = tbl[(tbl["u/s"] < 0.85) | (tbl["u/s"] > 1.15)]
    if not outliers.empty:
        print(f"  Outliers (>15% diff):")
        print(outliers.to_string(float_format=lambda x: f"{x:.2f}"))


  D. 4-bit fused (MatMulNBits kernel) — unsigned/signed ratio
  amd: min=0.85  max=1.05  median=0.96
  intel: min=0.81  max=0.97  median=0.95
  arm: min=0.75  max=1.39  median=1.00
  Outliers (>15% diff):
                                signed  unsigned  u/s
device model_size seq_length                         
arm    0.5b       128          2975.98   2242.95 0.75
       3b         1           67694.52  81430.93 1.20
                  128         69554.13  89847.95 1.29
                  256         68359.03  94954.56 1.39
                  512         82941.74 100875.96 1.22
                  1024       104074.25 131224.39 1.26
intel  7b         256         76477.57  62085.65 0.81

  D. 4-bit unfused (DequantizeLinear + MatMul) — unsigned/signed ratio
  amd: min=0.91  max=1.43  median=0.96
  intel: min=0.84  max=1.06  median=0.95
  arm: min=0.88  max=1.11  median=0.97
  Outliers (>15% diff):
                               signed  unsigned  u/s
device model_size seq_length            

### Observations

1. **4-bit fused (QDQ):** Output shows unsigned/signed ratio: AMD median 0.96 (0.85–1.05), Intel median 0.95 (0.81–0.97). Unsigned is ~4–5% faster (i.e., signed is slightly slower). In FP32 this was exactly 1.0 (fused to same MNB kernel). The small FP16 gap is because "fused" doesn't actually fuse — signed and unsigned run slightly different unfused DQ code paths.

2. **ARM 4-bit fused:** median 1.00 (0.75–1.39). Wide range includes ARM 3b outliers. No systematic trend.

3. **8-bit:** AMD/Intel/ARM all near 1.0 (median 0.99–1.02). No difference for byte-width types, same as FP32.

Conclusion: Signedness is a non-factor for 8-bit. The small 4-bit x86 gap (4–5% signed slower) is an artifact of the unfused DQ kernel's nibble extraction for int4 vs uint4, consistent with FP32's unfused path finding.

---

## E. QDQ Block vs Channel (Original Layout Only)

- **Block original**: Would fuse to MatMulNBits for 4-bit in FP32, but **does not fuse in FP16** (Section A)
- **Channel original**: Per-channel quantization, no block structure, cannot fuse in either FP16 or FP32

Note: No transpose models in FP16 dataset. `channel-sym-signed` models fail to load across all devices (ORT bug — same as FP32).

In [8]:
# Section E: Effect of granularity on QDQ performance
# Only 4-bit block_original fuses to MatMulNBits. Channel runs unfused DQ+MatMul.

qdq_fused = df[(df["format_type"] == "qdq") & (df["scenario"] == "qdq_fused")].copy()
qdq_fused["layout_label"] = qdq_fused["granularity"] + "_" + qdq_fused["weight_layout"]

# Show decode (seq=1) — where layout matters most
for bits in [4, 8]:
    sub = qdq_fused[(qdq_fused["bits"] == bits) & (qdq_fused["symmetry"] == "asym") &
                     (qdq_fused["signedness"] == "signed") & (qdq_fused["seq_length"] == 1)]
    pvt = sub.pivot_table(index=["device", "model_size"], columns="layout_label", values="mean_ms")

    note = "(block_original fuses to MatMulNBits)" if bits == 4 else "(no fusion for any layout)"
    print(f"\n  {bits}-bit decode (seq=1) {note}")
    print("  " + "-" * 90)
    print(pvt.to_string(float_format=lambda x: f"{x:.0f}"))

# Summarize seq=1024 gap (should be much smaller)
sub4_1024 = qdq_fused[(qdq_fused["bits"] == 4) & (qdq_fused["symmetry"] == "asym") &
                       (qdq_fused["signedness"] == "signed") & (qdq_fused["seq_length"] == 1024)]
pvt_1024 = sub4_1024.pivot_table(index=["device", "model_size"], columns="layout_label", values="mean_ms")
if "block_original" in pvt_1024.columns:
    ratios = []
    for col in [c for c in pvt_1024.columns if c != "block_original"]:
        r = pvt_1024[col] / pvt_1024["block_original"]
        ratios.extend(r.dropna().values)
    if ratios:
        print(f"\n  4-bit seq=1024: unfused/block_original ratio range = {min(ratios):.1f}-{max(ratios):.1f}x")
        print(f"  (Gap shrinks at prefill — compute dominates)")


  4-bit decode (seq=1) (block_original fuses to MatMulNBits)
  ------------------------------------------------------------------------------------------
layout_label       block_original  channel_original
device model_size                                  
amd    0.5b                  2872              3258
       1.5b                  7939             10792
       3b                   19779             23568
       7b                   45565               NaN
arm    0.5b                  2131              2633
       1.5b                  7642              9007
       3b                   67695             48583
       7b                  355212               NaN
intel  0.5b                  3364              3776
       1.5b                 10495             12913
       3b                   21129             25879
       7b                   60658             66626

  8-bit decode (seq=1) (no fusion for any layout)
  ----------------------------------------------------------------

### Observations

1. **4-bit block vs channel:** AMD median 1.21 (0.96–1.40), Intel median 1.15 (1.05–1.30), ARM median 1.08 (0.72–1.24). Channel is 8–21% slower than block on median. In FP32, block_original fused to MNB making it 51–209× faster than channel. In FP16, neither fuses, so the gap is just the DQ kernel overhead difference (block vs channel granularity).

2. The fact that block and channel are within 1.0–1.4× of each other confirms both run the same unfused DQ+MatMul path. If block were fusing, it would be 100×+ faster.

3. **8-bit block vs channel:** AMD median 1.02, Intel 1.02, ARM 1.02. Near parity — consistent with both running identical unfused 8-bit DQ paths.

4. **ARM 3b 4-bit outlier:** channel/block = 0.72 at seq=1 (channel faster). This inverts the trend and is likely anomalous data — ARM 3b consistently shows erratic values throughout the analysis.

Conclusion: In FP16, block vs channel granularity has minimal impact (≤21% for 4-bit, ~2% for 8-bit) because neither fuses. This contrasts sharply with FP32 where block_original was 51–209× faster due to fusion.

---

## F. Unfused DQ Kernel: Block vs Channel

Compare DequantizeLinear kernel performance between **block** and **channel** granularity when both run unfused DQ+MatMul.

**Data sources:**
- **4-bit block**: `qdq_unfused` scenario
- **4-bit channel**: `qdq_fused` scenario — unfused in practice (ORT has no channel-to-MatMulNBits rewrite)
- **8-bit block & channel**: both use `qdq_fused` scenario — neither fuses

Filtered to `original` layout, `asym`, `signed`.

In [9]:
# Unfused DQ Kernel: Block vs Channel
# Both paths run unfused DQ+MatMul — this isolates the DQ kernel cost difference.
# Filtered to original layout, asym, signed.

for bits in [4, 8]:
    # Block unfused: for 4-bit use qdq_unfused scenario; for 8-bit use qdq_fused (doesn't fuse)
    block_scenario = "qdq_unfused" if bits == 4 else "qdq_fused"
    block_unfused = df[
        (df["format_type"] == "qdq") & (df["bits"] == bits) & (df["granularity"] == "block") &
        (df["scenario"] == block_scenario) & (df["weight_layout"] == "original") &
        (df["symmetry"] == "asym") & (df["signedness"] == "signed")
    ][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "block_ms"})

    # Channel unfused: always qdq_fused (never fuses for any bit width)
    channel_unfused = df[
        (df["format_type"] == "qdq") & (df["bits"] == bits) & (df["granularity"] == "channel") &
        (df["scenario"] == "qdq_fused") & (df["weight_layout"] == "original") &
        (df["symmetry"] == "asym") & (df["signedness"] == "signed")
    ][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "channel_ms"})

    combined = block_unfused.merge(channel_unfused, on=["device", "model_size", "seq_length"])
    combined["ch/block"] = combined["channel_ms"] / combined["block_ms"]

    print(f"\n{'='*90}")
    block_note = "qdq_unfused" if bits == 4 else "qdq_fused (unfused in practice)"
    print(f"  Unfused DQ: Block vs Channel — {bits}-bit asym signed, original layout")
    print(f"  Block source: {block_note} | Channel source: qdq_fused (unfused in practice)")
    print(f"  ch/block > 1 = channel slower, < 1 = channel faster")
    print(f"{'='*90}")

    if combined.empty:
        print("  No matching data found.")
        continue

    # Per-device pivot: model_size (rows) × seq_length (cols)
    for dev in DEVICE_ORDER:
        sub = combined[combined["device"] == dev]
        if sub.empty:
            continue
        pvt = sub.pivot_table(index="model_size", columns="seq_length", values="ch/block")
        pvt = pvt.reindex(MODEL_SIZE_ORDER).dropna(how="all")
        print(f"\n  {dev.upper()} — ch/block ratio")
        print("  " + "-"*60)
        print(pvt.to_string(float_format=lambda x: f"{x:.2f}"))

    # Absolute latencies side by side for seq=1 (decode)
    seq1_dq = combined[combined["seq_length"] == 1].set_index(["device", "model_size"]).sort_index()
    if not seq1_dq.empty:
        print(f"\n  Absolute latency at seq=1 (ms):")
        print("  " + "-"*60)
        print(seq1_dq[["block_ms", "channel_ms", "ch/block"]].to_string(float_format=lambda x: f"{x:.1f}"))

    # Cross-device summary
    print(f"\n  Per-device summary ({bits}-bit):")
    for dev in DEVICE_ORDER:
        ratios = combined[combined["device"] == dev]["ch/block"]
        if ratios.empty:
            continue
        print(f"    {dev}: min={ratios.min():.2f}  max={ratios.max():.2f}  "
              f"median={ratios.median():.2f}  mean={ratios.mean():.2f}")


  Unfused DQ: Block vs Channel — 4-bit asym signed, original layout
  Block source: qdq_unfused | Channel source: qdq_fused (unfused in practice)
  ch/block > 1 = channel slower, < 1 = channel faster

  AMD — ch/block ratio
  ------------------------------------------------------------
seq_length  1     128   256   512   1024
model_size                              
0.5b        1.06  1.12  1.14  1.07  1.09
1.5b        1.32  1.30  1.30  1.26  1.20
3b          1.13  1.19  1.22  1.28  1.40

  INTEL — ch/block ratio
  ------------------------------------------------------------
seq_length  1     128   256   512   1024
model_size                              
0.5b        1.15  1.16  1.14  1.11  1.07
1.5b        1.28  1.25  1.21  1.31  1.15
3b          1.22  1.22  1.18  1.18  1.12
7b          1.13  1.16  1.10  1.10  1.11

  ARM — ch/block ratio
  ------------------------------------------------------------
seq_length  1     128   256   512   1024
model_size                              
0.5

### Observations

**4-bit block vs channel (unfused):**

1. AMD: channel 6–40% slower at seq=1 (1.06–1.32×). Median 1.20. Gap persists but narrows at longer sequences. Same direction as FP32 (7–18%).

2. Intel: channel 13–28% slower at seq=1 (1.13–1.28×). Median 1.15. Consistent across model sizes. In FP32, Intel showed a larger gap (42–58%).

3. ARM: channel 6–18% slower at seq=1 (1.06–1.18×). Moderate and consistent. ARM 3b doesn't show the outlier seen in Section E's fused path.

**8-bit block vs channel:**

4. x86: near parity. AMD median 1.02, Intel median 1.02. Range 0.93–1.20. No significant advantage to either granularity. Different from FP32 where 8-bit block was 2.8–5.1× faster at seq=1 — this is because FP16 block DQ itself is slower (offsetting any kernel advantage over channel).

5. ARM: median 1.02 (0.83–1.13). Slight advantage to block at seq=1 but near parity overall. ARM 3b doesn't show the 3.5–7.0× block advantage seen in FP32.

Conclusion: Block DQ is 6–40% faster than channel DQ for 4-bit (consistent with FP32 direction but smaller magnitude). For 8-bit, block and channel are at parity in FP16, unlike FP32 where block had a large advantage.

---

## G. Channel DQ: 4-bit vs 8-bit

Both 4-bit and 8-bit channel run unfused DQ+MatMul (no fusion possible). The DQ kernel is the only difference. 4-bit needs bit extraction, 8-bit reads full bytes.

In [10]:
# Section G: Channel DQ 4-bit vs 8-bit (both unfused)
# Both use qdq_fused scenario but neither actually fuses.
# Filtered to channel, original layout, asym, signed.

for dev in DEVICE_ORDER:
    ch4 = df[(df["format_type"]=="qdq") & (df["bits"]==4) & (df["granularity"]=="channel") &
             (df["scenario"]=="qdq_fused") & (df["weight_layout"]=="original") &
             (df["symmetry"]=="asym") & (df["signedness"]=="signed") &
             (df["device"]==dev)][["model_size","seq_length","mean_ms"]].rename(columns={"mean_ms":"ch4_ms"})
    ch8 = df[(df["format_type"]=="qdq") & (df["bits"]==8) & (df["granularity"]=="channel") &
             (df["scenario"]=="qdq_fused") & (df["weight_layout"]=="original") &
             (df["symmetry"]=="asym") & (df["signedness"]=="signed") &
             (df["device"]==dev)][["model_size","seq_length","mean_ms"]].rename(columns={"mean_ms":"ch8_ms"})
    m = ch4.merge(ch8, on=["model_size","seq_length"])
    if m.empty:
        print(f"\n  {dev.upper()}: No matching data for channel 4-bit vs 8-bit")
        continue
    m["8b/4b"] = m["ch8_ms"] / m["ch4_ms"]
    pvt = m.pivot_table(index="model_size", columns="seq_length", values="8b/4b")
    pvt = pvt.reindex(MODEL_SIZE_ORDER).dropna(how="all")
    print(f"\n  {dev.upper()} — channel 8b/4b ratio (< 1 = 8-bit faster)")
    print("  " + "-"*50)
    print(pvt.to_string(float_format=lambda x: f"{x:.2f}"))

    # seq=1 absolute values
    s1 = m[m["seq_length"]==1].sort_values("model_size")
    print(f"  seq=1 absolute (ms):")
    for _, row in s1.iterrows():
        print(f"    {row['model_size']:4s}  4b={row['ch4_ms']:.0f}  8b={row['ch8_ms']:.0f}  8b/4b={row['8b/4b']:.2f}")


  AMD — channel 8b/4b ratio (< 1 = 8-bit faster)
  --------------------------------------------------
seq_length  1     128   256   512   1024
model_size                              
0.5b        0.57  0.58  0.61  0.64  0.71
1.5b        0.57  0.59  0.60  0.64  0.70
3b          0.58  0.67  0.70  0.75  0.88
  seq=1 absolute (ms):
    0.5b  4b=3258  8b=1865  8b/4b=0.57
    1.5b  4b=10792  8b=6145  8b/4b=0.57
    3b    4b=23568  8b=13576  8b/4b=0.58

  INTEL — channel 8b/4b ratio (< 1 = 8-bit faster)
  --------------------------------------------------
seq_length  1     128   256   512   1024
model_size                              
0.5b        0.52  0.55  0.59  0.64  0.74
1.5b        0.50  0.51  0.55  0.55  0.68
3b          0.50  0.53  0.56  0.62  0.69
7b          1.19  1.12  1.13  1.03  0.98
  seq=1 absolute (ms):
    0.5b  4b=3776  8b=1974  8b/4b=0.52
    1.5b  4b=12913  8b=6393  8b/4b=0.50
    3b    4b=25879  8b=13028  8b/4b=0.50
    7b    4b=66626  8b=79573  8b/4b=1.19

  ARM — chann

### Observations

1. **x86 seq=1: 8-bit channel is faster than 4-bit channel.** AMD 8b/4b = 0.57–0.58, Intel 0.50–0.52. 8-bit is ~2× faster. Same direction as FP32 (0.52–0.82) but FP16 gap is larger. The nibble extraction overhead in the 4-bit DQ kernel costs more than the 2× larger 8-bit input.

2. **Intel 7b outlier:** 8b/4b = 1.19 at seq=1 (8-bit slower). This is the only 7b QDQ model — the model may be large enough to cause cache effects that reverse the trend. Narrows to 0.98 at seq=1024.

3. **x86 at higher seq:** gap narrows. AMD reaches 0.71–0.88 at seq=1024 (8-bit still faster). Intel 0.68–0.74 for 0.5b-3b. Consistent with FP32 findings.

4. **ARM 0.5b-1.5b:** 8-bit is also faster (0.46–0.72). Stronger advantage than FP32's ARM results. ARM's nibble extraction in FP16 appears more expensive than in FP32.

5. **ARM 3b:** 8-bit is 2× SLOWER (8b/4b = 1.19–1.99). The 3b 8-bit model triggers the memory pressure issues seen consistently throughout the ARM data. Not kernel-related.

Conclusion: On both x86 and ARM (when models fit in RAM), unfused 8-bit channel DQ is faster than 4-bit. FP16 amplifies this advantage vs FP32 — nibble extraction is relatively more expensive in the FP16 DQ path.

---

## H. Unfused DQ: 4-bit vs 8-bit (Block)

In [11]:
# Section H: Unfused 4-bit vs 8-bit — both run DQ+MatMul
# 4-bit: qdq_unfused scenario (explicitly unfused)
# 8-bit: qdq_fused scenario (doesn't actually fuse — fusion selector only matches 4-bit)
qdq_4_unfused = df[
    (df["format_type"] == "qdq") & (df["bits"] == 4) & (df["granularity"] == "block") &
    (df["scenario"] == "qdq_unfused") & (df["weight_layout"] == "original") &
    (df["symmetry"] == "asym") & (df["signedness"] == "signed")
][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "4bit_ms"})

qdq_8_unfused = df[
    (df["format_type"] == "qdq") & (df["bits"] == 8) & (df["granularity"] == "block") &
    (df["scenario"] == "qdq_fused") & (df["weight_layout"] == "original") &
    (df["symmetry"] == "asym") & (df["signedness"] == "signed")
][["device", "model_size", "seq_length", "mean_ms"]].rename(columns={"mean_ms": "8bit_ms"})

combined = qdq_4_unfused.merge(qdq_8_unfused, on=["device", "model_size", "seq_length"])
combined["8b/4b"] = combined["8bit_ms"] / combined["4bit_ms"]

for dev in DEVICE_ORDER:
    dev_df = combined[combined["device"] == dev]
    if dev_df.empty:
        print(f"\n  {dev.upper()}: No matching data")
        continue
    pvt = dev_df.pivot_table(index="model_size", columns="seq_length", values="8b/4b")
    pvt = pvt.reindex(MODEL_SIZE_ORDER).dropna(how="all")
    print(f"\n  {dev.upper()} — unfused 8b/4b ratio (< 1 = 8-bit faster)")
    print("  " + "-"*50)
    print(pvt.to_string(float_format=lambda x: f"{x:.2f}"))

    # Show absolute values at seq=1
    s1 = dev_df[dev_df["seq_length"] == 1].sort_values("model_size")
    print(f"  seq=1 absolute (ms):")
    for _, row in s1.iterrows():
        print(f"    {row['model_size']:4s}  4b={row['4bit_ms']:.0f}  8b={row['8bit_ms']:.0f}  8b/4b={row['8b/4b']:.2f}")

    # Show absolute values at seq=1024
    s1024 = dev_df[dev_df["seq_length"] == 1024].sort_values("model_size")
    print(f"  seq=1024 absolute (ms):")
    for _, row in s1024.iterrows():
        print(f"    {row['model_size']:4s}  4b={row['4bit_ms']:.0f}  8b={row['8bit_ms']:.0f}  8b/4b={row['8b/4b']:.2f}")


  AMD — unfused 8b/4b ratio (< 1 = 8-bit faster)
  --------------------------------------------------
seq_length  1     128   256   512   1024
model_size                              
0.5b        0.57  0.64  0.68  0.68  0.76
1.5b        0.66  0.69  0.71  0.75  0.79
3b          0.65  0.77  0.85  0.95  1.21
  seq=1 absolute (ms):
    0.5b  4b=3072  8b=1765  8b/4b=0.57
    1.5b  4b=8204  8b=5421  8b/4b=0.66
    3b    4b=20931  8b=13502  8b/4b=0.65
  seq=1024 absolute (ms):
    0.5b  4b=4536  8b=3452  8b/4b=0.76
    1.5b  4b=12577  8b=9873  8b/4b=0.79
    3b    4b=29450  8b=35683  8b/4b=1.21

  INTEL — unfused 8b/4b ratio (< 1 = 8-bit faster)
  --------------------------------------------------
seq_length  1     128   256   512   1024
model_size                              
0.5b        0.64  0.68  0.67  0.74  0.82
1.5b        0.62  0.64  0.65  0.72  0.78
3b          0.61  0.63  0.64  0.71  0.78
7b          1.15  1.09  1.08  0.98  0.97
  seq=1 absolute (ms):
    0.5b  4b=3282  8b=2111  8b

### Observations

**AMD:** 8-bit block is faster than 4-bit block at seq=1 (8b/4b = 0.57–0.66). Gap narrows toward seq=1024 (0.76–1.21). 3b at seq=1024 crosses 1.0 (4-bit becomes faster). Same direction as FP32 (0.21–0.31 at seq=1) but FP16 gap is smaller.

**Intel:** 8-bit faster at seq=1 for 0.5b-3b (0.61–0.64). 7b outlier: 1.15 (8-bit slower — only 7b QDQ model, likely cache/memory effects). Narrows to 0.78–0.97 at seq=1024. In FP32, Intel saw 0.21–0.26 at seq=1.

**ARM 0.5b-1.5b:** 8-bit faster (0.53–0.58 at seq=1). Same direction as x86. Narrows to 0.74–0.81 at seq=1024.

**ARM 3b:** 8-bit is 2.25× SLOWER at seq=1. Same memory pressure pattern as Sections C, E, and G. Crosses back below 1.0 by seq=256 and reaches 1.19 at seq=1024.

**FP16 vs FP32:** The 8-bit advantage at seq=1 is weaker in FP16 (0.53–0.66 vs 0.21–0.31 in FP32). This is because the FP16 8-bit DQ kernel is less optimized, partially negating the nibble-extraction advantage over 4-bit. On ARM, the FP16 8-bit overhead is even worse (see Section C's 67–124× issue).

Conclusion: 8-bit unfused block is still faster than 4-bit on x86 (same direction as FP32), but the advantage is ~half as large. ARM follows x86 for models that fit in RAM.

---

## I. Model Load Time

Session creation time across formats/layouts/devices. Includes graph optimization + weight preparation.

In [12]:
# Section I: Model load time
# One load time per model (not per seq_length), so deduplicate
load_times = df.drop_duplicates(
    subset=["device", "format_type", "model_size", "bits", "symmetry",
            "granularity", "signedness", "weight_layout", "scenario"]
)[["device", "format_type", "model_size", "bits", "symmetry", "granularity",
   "signedness", "weight_layout", "scenario", "model_load_time_s"]].copy()

load_times = load_times.sort_values(["device", "format_type", "weight_layout", "bits", "model_size"])

# I.1: MNB load times
mnb_load = load_times[load_times["format_type"] == "mnb"]
tbl = mnb_load.pivot_table(
    index=["bits", "symmetry", "model_size"],
    columns="device",
    values="model_load_time_s"
)
print(f"\n{'='*80}")
print(f"  I.1 MNB model load times (seconds)")
print(f"{'='*80}")
print(tbl.to_string(float_format=lambda x: f"{x:.2f}"))

# I.2: QDQ load times — focus on block fused, original
qdq_load = load_times[
    (load_times["format_type"] == "qdq") & (load_times["granularity"] == "block") &
    (load_times["scenario"] == "qdq_fused") & (load_times["signedness"] == "signed") &
    (load_times["symmetry"] == "asym")
]
tbl2 = qdq_load.pivot_table(
    index=["bits", "weight_layout", "model_size"],
    columns="device",
    values="model_load_time_s"
)
print(f"\n{'='*80}")
print(f"  I.2 QDQ block asym signed fused — load times (seconds)")
print(f"{'='*80}")
print(tbl2.to_string(float_format=lambda x: f"{x:.2f}"))

# I.3: Flag extreme load times (>30s)
extreme = load_times[load_times["model_load_time_s"] > 30]
if not extreme.empty:
    print(f"\n{'='*80}")
    print(f"  I.3 EXTREME load times (>30s) — possible issues")
    print(f"{'='*80}")
    print(extreme.to_string(index=False, float_format=lambda x: f"{x:.1f}"))
else:
    print("\nNo extreme load times (>30s) detected.")


  I.1 MNB model load times (seconds)
device                    amd  arm  intel
bits symmetry model_size                 
4    asym     0.5b       0.59 0.70   1.50
              1.5b       1.82 1.92   2.40
              3b         3.46 3.31   4.78
              7b         7.15 8.13  10.10
     sym      0.5b       0.73 0.72   0.89
              1.5b       1.72 1.95   2.39
              3b         3.10 3.37   4.39
              7b         6.69 8.13  11.33
8    asym     0.5b       0.87 0.19   1.16
              1.5b       2.18 0.22   3.50
              3b         4.08 0.27   6.28
              7b         8.91 0.26  13.16
     sym      0.5b       0.83 0.16   1.13
              1.5b       2.08 0.19   3.23
              3b         3.73 0.24   6.05
              7b         8.39 0.23  12.64

  I.2 QDQ block asym signed fused — load times (seconds)
device                         amd  arm  intel
bits weight_layout model_size                 
4    original      0.5b       0.25 0.25   0.33
       

### Observations

1. **MNB 4-bit load times match FP32 pattern.** AMD 0.59–7.15s, Intel 0.89–11.33s, ARM 0.70–8.13s. PrePack runs normally — MNB fusion works as expected. Scales linearly with model size.

2. **MNB 8-bit on ARM loads suspiciously fast.** 0.16–0.27s regardless of model size (0.5b to 7b). Compare to x86: AMD 0.83–8.91s, Intel 1.13–13.16s. This confirms the ARM 8-bit MNB FP16 kernel is NOT doing PrePack — no weight repacking at load time. This directly explains the 67–124× runtime slowdown from Section C.

3. **QDQ "fused" 4-bit loads much faster than MNB.** 0.25–0.47s vs MNB's 0.59–11.33s. No PrePack = no fusion. In FP32, fused QDQ load times matched MNB because both PrePacked.

4. **QDQ "fused" 8-bit loads fast on all devices.** 0.19–0.46s. Expected — 8-bit QDQ never fuses in either FP16 or FP32.

5. **Extreme channel QDQ load times (I.3):** Channel QDQ models take 34–164s to load vs <1s for block QDQ. Channel quantization stores per-output-channel scales and zero-points, creating many more small tensors for ORT to process during graph optimization. This is a graph-construction overhead, not a kernel issue.

6. **`channel-sym-signed` is absent from FP16 data entirely** (no models were collected for this configuration), so the NaN/load-failure issue from FP32 cannot be evaluated.

Conclusion: Load time is the diagnostic fingerprint for fusion. MNB 4-bit loads slow (PrePack), everything else loads fast. ARM MNB 8-bit fast load + slow runtime = missing PrePack optimization for this kernel path.

---

## J. Data Validation

In [13]:
# Section J: Data validation

print("="*80)
print("  J.1 Row counts per device")
print("="*80)
print(df.groupby("device").size().to_string())

print(f"\n{'='*80}")
print("  J.2 Breakdown by device × format_type × weight_layout × scenario")
print("="*80)
breakdown = df.groupby(["device", "format_type", "weight_layout", "scenario"]).size().unstack(fill_value=0)
print(breakdown.to_string())

print(f"\n{'='*80}")
print("  J.3 Configs per device (unique format+size+bits+sym+gran+sign+layout+scenario)")
print("="*80)
for device in DEVICE_ORDER:
    dev_df = df[df["device"] == device]
    configs = dev_df.drop_duplicates(
        subset=["format_type", "model_size", "bits", "symmetry", "granularity",
                "signedness", "weight_layout", "scenario"]
    )
    print(f"  {device}: {len(configs)} unique configs, {len(dev_df)} total rows")

# J.4: Check for duplicates after disambiguation
print(f"\n{'='*80}")
print("  J.4 Duplicate check (same device+config+seq_length)")
print("="*80)
group_cols = ["device", "format_type", "model_size", "bits", "symmetry",
              "granularity", "signedness", "weight_layout", "scenario", "seq_length"]
dupes = df.groupby(group_cols).size()
dupes_gt1 = dupes[dupes > 1]
if dupes_gt1.empty:
    print("  No duplicates found — data is clean!")
else:
    print(f"  {len(dupes_gt1)} duplicate groups found:")
    print(dupes_gt1.to_string())

# J.5: Missing configs across devices
print(f"\n{'='*80}")
print("  J.5 Configs present in one device but missing from another (seq=1 only)")
print("="*80)
config_cols = ["format_type", "model_size", "bits", "symmetry", "granularity",
               "signedness", "weight_layout", "scenario"]
decode_df = df[df["seq_length"] == 1]
for d1 in DEVICE_ORDER:
    for d2 in DEVICE_ORDER:
        if d1 >= d2:
            continue
        s1 = set(decode_df[decode_df["device"] == d1][config_cols].apply(tuple, axis=1))
        s2 = set(decode_df[decode_df["device"] == d2][config_cols].apply(tuple, axis=1))
        only_d1 = s1 - s2
        only_d2 = s2 - s1
        if only_d1 or only_d2:
            print(f"\n  {d1} vs {d2}:")
            if only_d1:
                print(f"    Only in {d1} ({len(only_d1)}):", list(only_d1)[:5], "..." if len(only_d1) > 5 else "")
            if only_d2:
                print(f"    Only in {d2} ({len(only_d2)}):", list(only_d2)[:5], "..." if len(only_d2) > 5 else "")

  J.1 Row counts per device
device
amd      285
arm      285
intel    340

  J.2 Breakdown by device × format_type × weight_layout × scenario
scenario                          native  qdq_fused  qdq_unfused
device format_type weight_layout                                
amd    mnb         original           80          0            0
       qdq         original            0        155           50
arm    mnb         original           80          0            0
       qdq         original            0        155           50
intel  mnb         original           80          0            0
       qdq         original            0        200           60

  J.3 Configs per device (unique format+size+bits+sym+gran+sign+layout+scenario)
  amd: 57 unique configs, 285 total rows
  intel: 68 unique configs, 340 total rows
  arm: 57 unique configs, 285 total rows

  J.4 Duplicate check (same device+config+seq_length)
  No duplicates found — data is clean!

  J.5 Configs present in one device 

### Observations

1. **Data is clean — zero NaN rows.** All 910 data points have valid latency values. This is simpler than FP32 which had NaN rows from `channel-sym-signed-transpose` failures.

2. **Row counts:** AMD 285, ARM 285, Intel 340. Intel has 55 extra rows (11 extra configs × 5 seq lengths) — all 7b QDQ models that AMD/ARM don't have.

3. **Config counts:** AMD/ARM: 57 configs each (identical coverage). Intel: 68 configs.

4. **No duplicates.** Every (device, config, seq_length) combination appears exactly once.

5. **`channel-sym-signed` was not collected** for FP16, so the FP32 load-failure issue cannot be compared.

6. **No transpose models** in FP16 data (only `original` layout). Transpose analysis is out of scope.

---

## K. FP16 vs FP32 Comparison

Cross-precision comparison: load FP32 data from `qdq-cpu/` and compare latencies with FP16 data.

**Notes:**
- FP32 data filtered to `weight_layout == "original"` only (to match FP16 scope)
- FP32 Intel folder is `surface_laptop_perf`; FP16 Intel is `intel_surface_laptop_perf`
- `fp16/fp32` ratio: < 1 means FP16 is faster, > 1 means FP32 is faster
- ORT versions differ slightly: FP16 AMD uses ORT 1.25.0 vs FP32 AMD ORT 1.24.2

In [14]:
# Load FP32 data
FP32_DATA_DIR = Path(r"..\..\..\data\qdq-cpu")
FP32_DEVICES = {
    "amd": FP32_DATA_DIR / "hp_amd_laptop_perf",
    "intel": FP32_DATA_DIR / "surface_laptop_perf",
    "arm": FP32_DATA_DIR / "surface_arm_laptop_perf",
}

fp32_frames = []
for device, path in FP32_DEVICES.items():
    csv_path = path / "summary.csv"
    if not csv_path.exists():
        print(f"WARNING: {csv_path} not found, skipping")
        continue
    tmp = pd.read_csv(csv_path)
    tmp["device"] = device
    fp32_frames.append(tmp)
    print(f"  FP32 {device}: {len(tmp)} rows")

df32 = pd.concat(fp32_frames, ignore_index=True)
df32["bits"] = df32["bits"].astype(int)
df32["seq_length"] = df32["seq_length"].astype(int)
df32["mean_ms"] = df32["mean_ms"].astype(float)
df32["model_load_time_s"] = pd.to_numeric(df32["model_load_time_s"], errors="coerce")

# Fill MNB granularity/signedness
mnb32_mask = df32["format_type"] == "mnb"
df32.loc[mnb32_mask, "granularity"] = "n/a"
df32.loc[mnb32_mask, "signedness"] = "n/a"

# Filter FP32 to original layout only (to match FP16 scope)
df32 = df32[df32["weight_layout"] == "original"].copy()

print(f"\nFP32 total rows (original layout only): {len(df32)}")
print(f"FP16 total rows: {len(df)}")

# Merge keys for comparison
merge_keys = ["device", "format_type", "model_size", "bits", "symmetry",
              "granularity", "signedness", "weight_layout", "scenario", "seq_length"]

merged_all = df.merge(df32, on=merge_keys, suffixes=("_fp16", "_fp32"), how="inner")
merged_all["fp16/fp32"] = merged_all["mean_ms_fp16"] / merged_all["mean_ms_fp32"]
print(f"Merged rows (inner join): {len(merged_all)}")

  FP32 amd: 640 rows
  FP32 intel: 640 rows
  FP32 arm: 560 rows

FP32 total rows (original layout only): 1140
FP16 total rows: 910
Merged rows (inner join): 910


In [15]:
# K.1: MNB FP16 vs FP32
print(f"{'='*90}")
print(f"  K.1 MNB FP16 vs FP32 — sym, original layout (fp16/fp32 ratio)")
print(f"{'='*90}")

mnb_merged = merged_all[merged_all["format_type"] == "mnb"]
for bits in [4, 8]:
    sub = mnb_merged[(mnb_merged["bits"] == bits) & (mnb_merged["symmetry"] == "sym")]
    if sub.empty:
        print(f"\n  {bits}-bit: No matching data")
        continue
    pvt = sub.pivot_table(index=["device", "model_size"], columns="seq_length", values="fp16/fp32")
    print(f"\n  {bits}-bit MNB sym — fp16/fp32 ratio (< 1 = FP16 faster):")
    print("  " + "-"*70)
    print(pvt.to_string(float_format=lambda x: f"{x:.2f}"))

    # Per-device summary
    for dev in DEVICE_ORDER:
        try:
            ratios = pvt.loc[dev].values.flatten()
            ratios = ratios[~np.isnan(ratios)]
            if len(ratios) > 0:
                print(f"  {dev}: min={ratios.min():.2f}  max={ratios.max():.2f}  median={np.median(ratios):.2f}")
        except KeyError:
            pass

  K.1 MNB FP16 vs FP32 — sym, original layout (fp16/fp32 ratio)

  4-bit MNB sym — fp16/fp32 ratio (< 1 = FP16 faster):
  ----------------------------------------------------------------------
seq_length         1     128   256   512   1024
device model_size                              
amd    0.5b        1.25  1.14  1.33  1.39  1.34
       1.5b        1.09  1.12  1.24  1.35  0.82
       3b          0.91  0.80  0.87  0.83  0.84
       7b          0.96  0.91  0.80  0.73  0.74
arm    0.5b        0.46  0.66  0.79  0.71  0.92
       1.5b        0.53  1.13  1.28  1.32  1.42
       3b          0.84  1.38  1.53  1.56  1.59
       7b          1.35  2.41  2.19  2.33  2.16
intel  0.5b        1.15  1.39  1.38  1.40  1.39
       1.5b        1.20  1.29  1.26  1.30  1.31
       3b          1.19  1.26  1.25  1.25  1.35
       7b          1.14  1.17  1.14  1.10  1.11
  amd: min=0.73  max=1.39  median=0.94
  intel: min=1.10  max=1.40  median=1.25
  arm: min=0.46  max=2.41  median=1.34

  8-bit MNB sym

In [16]:
# K.2: QDQ Fused (4-bit block) FP16 vs FP32
print(f"{'='*90}")
print(f"  K.2 QDQ Fused (4-bit block asym signed) — FP16 vs FP32")
print(f"{'='*90}")

fused_merged = merged_all[
    (merged_all["format_type"] == "qdq") & (merged_all["bits"] == 4) &
    (merged_all["granularity"] == "block") & (merged_all["scenario"] == "qdq_fused") &
    (merged_all["symmetry"] == "asym") & (merged_all["signedness"] == "signed")
]
if fused_merged.empty:
    print("  No matching data")
else:
    pvt = fused_merged.pivot_table(index=["device", "model_size"], columns="seq_length", values="fp16/fp32")
    print(pvt.to_string(float_format=lambda x: f"{x:.2f}"))
    for dev in DEVICE_ORDER:
        try:
            ratios = pvt.loc[dev].values.flatten()
            ratios = ratios[~np.isnan(ratios)]
            if len(ratios) > 0:
                print(f"  {dev}: min={ratios.min():.2f}  max={ratios.max():.2f}  median={np.median(ratios):.2f}")
        except KeyError:
            pass

  K.2 QDQ Fused (4-bit block asym signed) — FP16 vs FP32
seq_length           1      128   256   512   1024
device model_size                                 
amd    0.5b        345.68  13.31  7.07  4.17  2.36
       1.5b        386.24  13.43  7.01  3.74  2.19
       3b          482.81  16.31  8.46  4.55  2.61
       7b          578.15  18.16 10.38  6.64  5.78
arm    0.5b        188.24  11.06  4.06  2.41  1.70
       1.5b        227.77  10.31  5.09  3.19  1.95
       3b         1199.90  42.58 22.53 13.38  7.90
       7b         6012.08 140.70 76.14 38.53 19.36
intel  0.5b        401.38   9.79  5.00  3.13  2.12
       1.5b        461.93  11.15  5.99  3.45  2.17
       3b          475.52  12.10  6.40  3.80  2.36
       7b          602.20  14.89  9.61  5.73  3.53
  amd: min=2.19  max=578.15  median=7.76
  intel: min=2.12  max=602.20  median=6.19
  arm: min=1.70  max=6012.08  median=16.37


In [17]:
# K.3: QDQ Unfused (4-bit block) FP16 vs FP32
print(f"{'='*90}")
print(f"  K.3 QDQ Unfused (4-bit block asym signed) — FP16 vs FP32")
print(f"{'='*90}")

unfused_merged = merged_all[
    (merged_all["format_type"] == "qdq") & (merged_all["bits"] == 4) &
    (merged_all["granularity"] == "block") & (merged_all["scenario"] == "qdq_unfused") &
    (merged_all["symmetry"] == "asym") & (merged_all["signedness"] == "signed")
]
if unfused_merged.empty:
    print("  No matching data")
else:
    pvt = unfused_merged.pivot_table(index=["device", "model_size"], columns="seq_length", values="fp16/fp32")
    print(pvt.to_string(float_format=lambda x: f"{x:.2f}"))
    for dev in DEVICE_ORDER:
        try:
            ratios = pvt.loc[dev].values.flatten()
            ratios = ratios[~np.isnan(ratios)]
            if len(ratios) > 0:
                print(f"  {dev}: min={ratios.min():.2f}  max={ratios.max():.2f}  median={np.median(ratios):.2f}")
        except KeyError:
            pass

  K.3 QDQ Unfused (4-bit block asym signed) — FP16 vs FP32
seq_length         1     128   256   512   1024
device model_size                              
amd    0.5b        4.44  3.27  2.66  2.10  1.51
       1.5b        3.62  2.88  2.47  1.94  1.44
       3b          4.77  3.49  2.86  2.15  1.50
       7b          5.83  4.82  4.01  3.41  3.95
arm    0.5b        3.63  2.69  2.31  1.83  1.55
       1.5b        4.26  3.23  2.79  2.08  1.55
       3b         13.20 11.38  9.81 10.33  6.55
       7b         35.51 31.23 27.01 20.00  9.16
intel  0.5b        3.33  2.77  2.50  2.14  1.65
       1.5b        3.17  2.71  2.48  2.10  1.69
       3b          3.38  2.91  2.63  2.17  1.74
       7b          4.15  3.58  3.43  3.01  2.46
  amd: min=1.44  max=5.83  median=3.08
  intel: min=1.65  max=4.15  median=2.67
  arm: min=1.55  max=35.51  median=5.41


In [18]:
# K.4: 8-bit Block FP16 vs FP32 (unfused on both — qdq_fused scenario doesn't fuse for 8-bit)
print(f"{'='*90}")
print(f"  K.4 8-bit Block (asym signed, qdq_fused scenario) — FP16 vs FP32")
print(f"{'='*90}")

block8_merged = merged_all[
    (merged_all["format_type"] == "qdq") & (merged_all["bits"] == 8) &
    (merged_all["granularity"] == "block") & (merged_all["scenario"] == "qdq_fused") &
    (merged_all["symmetry"] == "asym") & (merged_all["signedness"] == "signed")
]
if block8_merged.empty:
    print("  No matching data")
else:
    pvt = block8_merged.pivot_table(index=["device", "model_size"], columns="seq_length", values="fp16/fp32")
    print(pvt.to_string(float_format=lambda x: f"{x:.2f}"))
    for dev in DEVICE_ORDER:
        try:
            ratios = pvt.loc[dev].values.flatten()
            ratios = ratios[~np.isnan(ratios)]
            if len(ratios) > 0:
                print(f"  {dev}: min={ratios.min():.2f}  max={ratios.max():.2f}  median={np.median(ratios):.2f}")
        except KeyError:
            pass

  K.4 8-bit Block (asym signed, qdq_fused scenario) — FP16 vs FP32
seq_length         1     128   256   512   1024
device model_size                              
amd    0.5b        9.44  4.71  3.26  2.07  1.40
       1.5b        9.10  4.59  3.19  2.07  1.39
       3b         11.15  6.01  4.28  2.87  2.19
arm    0.5b       10.21  3.79  2.45  1.71  1.25
       1.5b        7.33  3.65  2.77  1.93  1.36
       3b         90.78 46.85 29.69 17.01  3.53
intel  0.5b       10.09  4.96  3.34  2.34  1.70
       1.5b        8.79  4.94  3.54  2.46  1.79
       3b          7.76  4.69  3.35  2.39  1.75
       7b         19.83 10.57  7.60  4.58  3.10
  amd: min=1.39  max=11.15  median=3.26
  intel: min=1.70  max=19.83  median=4.06
  arm: min=1.25  max=90.78  median=3.65


In [19]:
# K.5: Channel FP16 vs FP32 (both 4-bit and 8-bit)
print(f"{'='*90}")
print(f"  K.5 Channel (asym signed, qdq_fused scenario) — FP16 vs FP32")
print(f"{'='*90}")

for bits in [4, 8]:
    ch_merged = merged_all[
        (merged_all["format_type"] == "qdq") & (merged_all["bits"] == bits) &
        (merged_all["granularity"] == "channel") & (merged_all["scenario"] == "qdq_fused") &
        (merged_all["symmetry"] == "asym") & (merged_all["signedness"] == "signed")
    ]
    if ch_merged.empty:
        print(f"\n  {bits}-bit channel: No matching data")
        continue
    pvt = ch_merged.pivot_table(index=["device", "model_size"], columns="seq_length", values="fp16/fp32")
    print(f"\n  {bits}-bit channel — fp16/fp32 ratio:")
    print("  " + "-"*70)
    print(pvt.to_string(float_format=lambda x: f"{x:.2f}"))
    for dev in DEVICE_ORDER:
        try:
            ratios = pvt.loc[dev].values.flatten()
            ratios = ratios[~np.isnan(ratios)]
            if len(ratios) > 0:
                print(f"  {dev}: min={ratios.min():.2f}  max={ratios.max():.2f}  median={np.median(ratios):.2f}")
        except KeyError:
            pass

  K.5 Channel (asym signed, qdq_fused scenario) — FP16 vs FP32

  4-bit channel — fp16/fp32 ratio:
  ----------------------------------------------------------------------
seq_length         1     128   256   512   1024
device model_size                              
amd    0.5b        4.24  3.77  3.19  2.86  2.30
       1.5b        4.46  3.89  3.41  2.90  2.24
       3b          4.92  4.32  3.98  3.43  2.66
arm    0.5b        4.26  3.31  2.78  2.28  1.75
       1.5b        3.37  2.78  2.40  1.91  1.81
       3b         13.21 10.74 10.46  9.34  6.55
intel  0.5b        2.43  2.34  2.13  1.91  1.55
       1.5b        2.83  2.59  2.38  2.27  1.72
       3b          2.86  2.85  2.60  2.26  1.87
       7b          3.32  3.20  2.90  2.71  2.41
  amd: min=2.24  max=4.92  median=3.43
  intel: min=1.55  max=3.32  median=2.42
  arm: min=1.75  max=13.21  median=3.31

  8-bit channel — fp16/fp32 ratio:
  ----------------------------------------------------------------------
seq_length         1   

In [20]:
# K.6: Load time FP16 vs FP32
print(f"{'='*90}")
print(f"  K.6 Model Load Time — FP16 vs FP32")
print(f"{'='*90}")

# Deduplicate load times (one per model config, not per seq_length)
dedup_cols = ["device", "format_type", "model_size", "bits", "symmetry",
              "granularity", "signedness", "weight_layout", "scenario"]

load_merged = merged_all.drop_duplicates(subset=dedup_cols)[
    dedup_cols + ["model_load_time_s_fp16", "model_load_time_s_fp32"]
].copy()
load_merged["fp16/fp32"] = load_merged["model_load_time_s_fp16"] / load_merged["model_load_time_s_fp32"]

# MNB load times
mnb_load_cmp = load_merged[load_merged["format_type"] == "mnb"]
if not mnb_load_cmp.empty:
    pvt = mnb_load_cmp.pivot_table(
        index=["bits", "symmetry", "model_size"],
        columns="device",
        values="fp16/fp32"
    )
    print(f"\n  MNB load time fp16/fp32 ratio:")
    print(pvt.to_string(float_format=lambda x: f"{x:.2f}"))

# QDQ block fused load times
qdq_load_cmp = load_merged[
    (load_merged["format_type"] == "qdq") & (load_merged["granularity"] == "block") &
    (load_merged["scenario"] == "qdq_fused") & (load_merged["signedness"] == "signed") &
    (load_merged["symmetry"] == "asym")
]
if not qdq_load_cmp.empty:
    pvt2 = qdq_load_cmp.pivot_table(
        index=["bits", "model_size"],
        columns="device",
        values="fp16/fp32"
    )
    print(f"\n  QDQ block asym signed fused — load time fp16/fp32 ratio:")
    print(pvt2.to_string(float_format=lambda x: f"{x:.2f}"))

  K.6 Model Load Time — FP16 vs FP32

  MNB load time fp16/fp32 ratio:
device                    amd  arm  intel
bits symmetry model_size                 
4    asym     0.5b       0.75 1.42   1.13
              1.5b       0.99 0.54   1.07
              3b         0.59 0.51   1.05
              7b         0.72 0.64   1.04
     sym      0.5b       0.86 0.39   0.78
              1.5b       0.80 0.43   0.83
              3b         0.45 0.42   0.77
              7b         0.71 0.44   0.92
8    asym     0.5b       1.00 0.06   0.99
              1.5b       0.57 0.03   1.24
              3b         0.60 0.02   1.04
              7b         0.70 0.01   0.98
     sym      0.5b       0.85 0.06   0.89
              1.5b       0.49 0.02   0.94
              3b         0.50 0.02   0.87
              7b         0.63 0.00   0.81

  QDQ block asym signed fused — load time fp16/fp32 ratio:
device           amd  arm  intel
bits model_size                 
4    0.5b       0.23 0.15   0.27
     1.5b     

### K — Cross-Comparison Observations

**K.1 MNB FP16 vs FP32 (cross-device synthesis):**
- Intel: FP16 is 1.1–1.5× slower than FP32 for 4-bit. Expected overhead from fp16 accumulation.
- AMD: mixed (0.7–1.3×), no clear trend.
- ARM 4-bit: near parity (0.9–1.5×). Reasonable.
- **ARM 8-bit: FP16 is 23–40× slower at seq=1.** The ARM 8-bit MNB FP16 kernel issue (Section C) shows up here as a massive FP16/FP32 gap. FP32 8-bit MNB on ARM works fine.

- **Cross-device summary for 4-bit MNB (the only fast path):** AMD shows near-parity or slight FP16 advantage for larger models (3b, 7b at 0.73–0.96×). Intel shows consistent FP16 penalty (1.1–1.4×). ARM 4-bit is mixed but reasonable (0.46–1.59×). For production FP16 MNB deployments, AMD is the best-performing platform.

**K.2 QDQ Fused FP16 vs FP32:**
- **FP16/FP32 ratio = 345–6012× at seq=1.** This is the quantitative proof that FP16 fusion is broken. FP32 fuses to MNB (fast), FP16 falls back to naïve DQ+MatMul (slow). The ratio IS the fusion speedup.
- At seq=1024: 2–20×. The unfused path is relatively cheaper at prefill but still significantly worse.

**K.3 QDQ Unfused FP16 vs FP32:**
- AMD: 1.4–5.8×. FP16 unfused DQ is slower than FP32 unfused DQ — the FP16 DQ kernel is less optimized.
- Intel: 1.7–4.2×. Similar magnitude.
- ARM: 1.6–35.5×. ARM 3b/7b show extreme ratios (10–35×) from compounded FP16 kernel overhead + memory pressure.
- Median: AMD 3.1, Intel 2.7, ARM 5.4. FP16 unfused DQ is 2.7–5.4× slower than FP32 across devices.

**K.4 8-bit Block FP16 vs FP32:**
- AMD/Intel 0.5b-3b: 7–11× at seq=1 (FP16 much slower). Narrows to 1.4–2.2× at seq=1024.
- Intel 7b: 20× at seq=1 (largest model, additional cache effects).
- **ARM 3b: 91× at seq=1.** The ARM 8-bit kernel issue from C compounds with the general FP16 DQ overhead.
- This section confirms that FP16 8-bit DQ kernels need significant optimization work.

**K.5 Channel FP16 vs FP32:**
- 4-bit channel: AMD 2.2–4.9×, Intel 1.6–3.3×, ARM 1.8–13.2×. Moderate overhead from FP16 channel DQ.
- 8-bit channel: AMD 1.6–3.6×, Intel 1.3–5.7× (7b outlier at 5.7), ARM 1.1–24.1× (3b = 24×). ARM 3b continues to show extreme FP16 degradation.
- 8-bit channel FP16/FP32 gap is smaller than 4-bit on ARM (for models that fit in RAM), suggesting byte-width DQ is relatively more optimized in both FP16 and FP32.

**K.6 Load Time FP16 vs FP32:**
- MNB 4-bit: fp16/fp32 ratio 0.4–1.4×. Similar load times — both PrePack normally.
- MNB 8-bit ARM: fp16/fp32 ratio ≈ 0.00–0.06. FP16 loads nearly instantly (no PrePack), FP32 takes seconds (PrePack runs). Confirms the ARM FP16 8-bit MNB PrePack is missing.
- QDQ fused 4-bit: fp16/fp32 = 0.01–0.27. FP16 loads much faster because it doesn't fuse (no PrePack). FP32 fuses and does PrePack.
- QDQ fused 8-bit: fp16/fp32 ≈ 1.0–2.1. Neither FP16 nor FP32 fuses 8-bit, so load times are comparable.

# Summary of Findings — FP16 QDQ CPU EP Performance

## Critical Finding: FP16 QDQ Fusion Is Broken

The single most important finding is that **DQ+MatMul → MatMulNBits fusion does NOT work for FP16 accumulation models**.

- In FP32, the `qdq_fused` scenario produces latencies nearly identical to native MNB (ratio ≈ 1.0×).
- In FP16, the `qdq_fused` scenario is **300–3 000× slower than MNB** at seq=1 and still **5–20× slower** at seq=1024.
- Likely root cause: the `DQMatMulToMatMulNBits` graph rewrite requires fp32 `DequantizeLinear` outputs, but FP16 models produce fp16 outputs, so the pattern never matches.
- Load-time evidence confirms this: FP16 "fused" models load **0.01–0.27× as fast** as MNB (no PrePack overhead), while FP32 fused models load on par with MNB.

## Section-by-Section Summary

| Section | Key Observations |
|---------|-----------------|
| **A – MNB vs QDQ** | MNB is the only performant path; both fused and unfused QDQ are 300–3 000× slower. No meaningful fused/unfused distinction (both fall back to naïve DQ+MatMul). |
| **B – MNB sym vs asym** | Ratios close to 1.0× across AMD/Intel. ARM shows 0.97–1.11× (within noise). Symmetry type does not meaningfully affect FP16 MNB performance. |
| **C – MNB 4-bit vs 8-bit** | AMD/Intel: 8-bit is 1.3–1.8× slower than 4-bit (expected). **ARM 8-bit is catastrophically slow at seq=1**: 67–124× slower for 0.5b–3b models. Improves toward 1.3× at seq=1024. Suggests ARM 8-bit MNB triggers a very slow kernel at short sequences. |
| **D – QDQ signed vs unsigned** | Ratio ≈ 1.0× for most configs. Minor ARM 3b outliers (up to 1.35× for 4-bit). 8-bit is near 1.0× everywhere. Signedness has negligible impact. |
| **E – QDQ block vs channel (fused)** | Neither block nor channel fuses in FP16. Channel is 8–21% slower than block (ch/block median 1.08–1.21) due to DQ kernel overhead differences. Much smaller gap than FP32 where fusion made block 51–209× faster. |
| **F – QDQ block vs channel (unfused)** | Block DQ is 6–40% faster than channel DQ for 4-bit (consistent direction, smaller magnitude than FP32). For 8-bit, block and channel are at parity. |
| **G – Channel DQ 4-bit vs 8-bit** | 8-bit channel is faster than 4-bit (8b/4b = 0.46–0.72). Counter-intuitive because 8-bit DQ processes more data per weight but accesses memory more sequentially. ARM 3b is an outlier (unstable data). |
| **H – Block DQ 4-bit vs 8-bit (unfused)** | 8-bit block is faster than 4-bit (8b/4b = 0.53–0.66), consistent with Section G. ARM 3b is an outlier. |
| **I – Model Load Time** | MNB loads slow (0.59–11.3s) because PrePack runs. QDQ "fused" loads fast (0.20–0.47s) because fusion never fires (no PrePack). ARM 8-bit MNB loads suspiciously fast (0.16–0.27s) — confirms missing PrePack for this kernel path. |
| **J – Data Validation** | Data is clean — zero NaN rows, no duplicates. `channel-sym-signed` models were not collected (failed to load on all devices). AMD/ARM have 57 configs, Intel 68 (extra 7b QDQ variants). |

## FP16 vs FP32 Cross-Comparison (Section K)

| Sub-section | Key Observations |
|-------------|-----------------|
| **K.1 – MNB** | Intel FP16 is 1.1–1.5× slower than FP32. AMD is mixed (0.7–1.3×). **ARM 8-bit MNB fp16/fp32 = 23–40× at seq=1** (ARM 8-bit kernel issue amplified in FP16). |
| **K.2 – QDQ Fused** | FP16/FP32 ratios of **345–6 012× at seq=1** because FP32 fuses to MNB but FP16 does not. This is the quantitative proof that FP16 fusion is broken. |
| **K.3 – QDQ Unfused** | FP16 is 1.4–5.8× slower than FP32 (AMD/Intel). **ARM 3b/7b reach 10–35×**. The unfused path itself is slower in FP16, likely due to fp16 DQ compute overhead. |
| **K.4 – 8-bit Block** | FP16/FP32 ratios of 7–11× at seq=1 (AMD/Intel). **ARM 3b = 91× at seq=1**. Even at seq=1024, FP16 is still 1.2–3.5× slower. |
| **K.5 – Channel** | 4-bit channel: 2–5× FP16/FP32 (AMD/Intel), ARM 3b 6–13×. 8-bit channel: 1.3–3.6× (AMD/Intel), **ARM 3b = 7–24×**. |
| **K.6 – Load Time** | MNB load times are comparable (fp16/fp32 ≈ 0.4–1.4×) except ARM 8-bit MNB which loads 50–100× faster in FP16 (trivially small). 4-bit QDQ fused loads 3–50× faster in FP16 (no PrePack), but 8-bit QDQ fused loads ~1–2× (FP32 8-bit also doesn't fuse well). |

## Actionable Takeaway

**For FP16 models on CPU EP today:** Use MNB native format exclusively. QDQ models are 300–3,000× slower due to broken fusion. There is no workaround — the graph rewrite must be fixed in ORT.

## Recommendations

1. **Fix FP16 QDQ fusion**: The `DQMatMulToMatMulNBits` rewrite must be extended to match FP16 DequantizeLinear outputs. This is the highest-priority fix — without it, QDQ FP16 models are unusable on CPU EP.
2. **Investigate ARM 8-bit MNB kernel**: The 67–124× slowdown at short sequences suggests a fallback to a naïve kernel. This affects both FP16 and FP32 (though FP16 is worse).
3. **Fix `channel-sym-signed`**: This configuration fails to load on all devices for both FP16 and FP32 models — an ORT bug in handling symmetric signed channel quantization.
4. **ARM 3b+ models under FP16**: ARM consistently shows 10–90× slowdowns for 3b/7b models in FP16 vs FP32. This warrants profiling to identify whether it's a DQ kernel issue or memory bandwidth bottleneck.